# Micro-KDN Evaluation Notebook (Colab)

This notebook helps you evaluate model predictions from your implementation using the repository's evaluation logic.

## Recommended Colab runtime
- **Runtime type:** Python 3
- **Hardware accelerator:** **CPU** (recommended for evaluation-only workflows)
- If you also want to run inference in the same notebook, switch to **T4 GPU**.

## 1) Clone repo and install dependencies
If you already mounted Drive and have the repo, update `REPO_URL` or skip cloning.

In [ ]:
REPO_URL = "https://github.com/<your-org>/<your-repo>.git"  # <-- change this
REPO_DIR = "/content/slm-kdn"

!git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install matplotlib pandas seaborn

## 2) Point to your predictions file
Your JSONL should contain at least:
- `intent`
- `target_command`
- `prediction`
- optional: `category`

In [ ]:
from pathlib import Path

PRED_FILE = Path("results/predictions/predictions.jsonl")  # <-- update if needed
assert PRED_FILE.exists(), f"Missing file: {PRED_FILE}"
PRED_FILE

## 3) Run evaluation (same metrics as `src/evaluate.py`)
Produces:
- `results/metrics/eval_metrics.json`

In [ ]:
!python src/evaluate.py --pred_file "$PRED_FILE" --out_dir results/metrics --out_file results/metrics/eval_metrics.json

In [ ]:
import json
from pprint import pprint

with open("results/metrics/eval_metrics.json", "r", encoding="utf-8") as f:
    metrics = json.load(f)

print("Overall metrics")
pprint(metrics["overall"])

## 4) Run error analysis
Produces:
- `results/error_analysis/error_summary.json`
- `results/error_analysis/errors.csv`

In [ ]:
!python src/error_analysis.py   --pred_file "$PRED_FILE"   --out_json results/error_analysis/error_summary.json   --out_csv results/error_analysis/errors.csv

In [ ]:
import json
import pandas as pd

with open("results/error_analysis/error_summary.json", "r", encoding="utf-8") as f:
    err = json.load(f)

err_df = (
    pd.DataFrame(err["counts"].items(), columns=["error_type", "count"])
      .sort_values("count", ascending=False)
)
err_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,4))
sns.barplot(data=err_df, x="error_type", y="count")
plt.xticks(rotation=30, ha="right")
plt.title("Error Type Distribution")
plt.tight_layout()
plt.show()

## 5) Inspect failure cases

In [ ]:
errors_csv = "results/error_analysis/errors.csv"
df_err = pd.read_csv(errors_csv)

# Show the most common non-correct errors
df_err[df_err["error_type"] != "correct"].head(20)

## 6) Optional: per-category leaderboard view

In [ ]:
per_cat = metrics.get("per_category", {})
if per_cat:
    cat_df = pd.DataFrame(per_cat).T.sort_values("normalized_exact_match", ascending=False)
    display(cat_df)
else:
    print("No category field found in predictions.")